# Week 6 Optional Extra - Deep Neural Network

Đây là phần bổ sung để "lấy lại mặt mũi" sau kết quả không như mong đợi.

Tôi đã huấn luyện Deep Neural Network trong file `pricer/deep_neural_network.py` và đã upload trọng số ở đây. Hãy tải file này vào thư mục week6:

File `deep_neural_network.pth` tại đây:

https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

Mục tiêu là thử nghiệm một giải pháp mạnh hơn dựa trên deep learning để dự đoán giá sản phẩm từ mô tả. Nếu mô hình trước đó chưa tốt, đây là cách để xem liệu một kiến trúc sâu hơn có khắc phục được khó khăn đó hay không.

### Tóm tắt quy trình của notebook

### Ý nghĩa chính của notebook

Notebook này dùng mô hình mạng nơ-ron sâu để cải thiện kết quả định giá sản phẩm, sau khi các phương pháp trước đó chưa đạt hiệu quả tốt. Ta tải dữ liệu, khởi tạo runner, nạp trọng số đã huấn luyện, rồi đánh giá mô hình trên tập test.

In [ ]:
from dotenv import load_dotenv
import os
from huggingface_hub import login
from pricer.evaluator import evaluate
from pricer.deep_neural_network import DeepNeuralNetworkRunner
from pricer.items import Item

# `DeepNeuralNetworkRunner` là lớp dùng để khởi tạo và chạy mạng nơ-ron sâu, còn `Item` giúp xử lý dữ liệu sản phẩm.
# Cell này nhập các thư viện và module cần thiết để tải dữ liệu, đăng nhập Hugging Face và chạy mô hình deep learning.

In [ ]:
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

# Việc đăng nhập này rất quan trọng để tải dữ liệu từ hub về máy local.

# Cell này cấu hình môi trường chạy notebook.# `load_dotenv()` đọc các biến môi trường từ file .env, sau đó `HF_TOKEN` được dùng để đăng nhập vào Hugging Face.
# `LITE_MODE = False` nghĩa là dùng toàn bộ dataset, không rút gọn.

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# Thông tin in ra giúp ta xác nhận dữ liệu đã được nạp đúng số lượng trước khi bắt đầu training hoặc inference.

# Cell này tải dữ liệu từ Hugging Face và chia thành 3 tập: train, validation và test.# `Item.from_hub(dataset)` trả về dữ liệu cần thiết để huấn luyện và đánh giá mô hình.

In [ ]:
runner = DeepNeuralNetworkRunner(train, val[:1000])
runner.setup()

# `setup()` chuẩn bị toàn bộ cấu hình mô hình và các thành phần cần thiết trước khi chạy training hoặc inference.

# Cell này khởi tạo runner cho mô hình deep neural network.# `train` là tập huấn luyện, còn `val[:1000]` là tập validation được cắt gọn để test nhanh.

## Nếu bạn muốn tự huấn luyện mô hình này

Hãy chạy đoạn sau - trên M1 Mac của tác giả, nó mất khoảng 4 giờ và sử dụng GPU mạnh:

```python
runner.train(epochs=5)
runner.save('deep_neural_network.pth')
```

## Hoặc chỉ cần tải file `deep_neural_network.pth` ở đây:

https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

Sau đó đặt file vào thư mục week6.

Đây là bước cải thiện hiệu quả mô hình bằng cách tận dụng mạng nơ-ron sâu, đặc biệt phù hợp khi dữ liệu đã được tiền xử lý và cần học các quan hệ phức tạp hơn giữa mô tả và giá.

### Tóm tắt quy trình của notebook

### Ý nghĩa chính của notebook

Notebook này cho phép bạn nạp trọng số đã được huấn luyện sẵn hoặc tự train lại mô hình nếu muốn. Mục tiêu là dùng kiến trúc deep neural network để dự đoán giá tốt hơn so với các phương pháp khác.

In [ ]:
#runner.load('deep_neural_network.pth')                          # nếu chạy trên MAC
runner.load('deep_neural_network.pth', 'cpu')                   # dùng dòng này nếu chạy trên máy Windows; nếu dùng dòng này thì comment dòng trên

# Nếu chạy trên Mac, có thể dùng cách load khác và không cần parameter `cpu`.

# Cell này nạp trọng số đã huấn luyện vào mô hình để sử dụng lại.# `cpu` được dùng cho môi trường Windows vì nhiều máy không có GPU hoặc không muốn chạy trên GPU.

In [ ]:
def deep_neural_network(item):
    return runner.inference(item)

evaluate(deep_neural_network, test)


# Hàm `deep_neural_network` lấy một sản phẩm và gọi `runner.inference(item)` để ước lượng giá.# Đây là bước cuối cùng để kiểm tra xem deep learning có thực sự cải thiện kết quả hay không.
# Sau đó, `evaluate(...)` so sánh các dự đoán trên tập test với giá thực tế để đánh giá hiệu suất của mô hình.

In [ ]:
# Cell trống này thường được giữ lại để người học có thể thử thêm các phép đo hoặc chạy các lần kiểm tra bổ sung.
# Bạn có thể thêm các câu lệnh đánh giá khác như in ra dự đoán cho một item cụ thể hoặc so sánh với mô hình baseline.

In [ ]:
# Cell này cũng có thể dùng để thử nghiệm thêm các mô hình hoặc ghi chú kết quả cuối cùng.
# Trong thực tế, đây là nơi bạn thường lưu ý các phân tích, so sánh chất lượng và quyết định mô hình nào nên chọn.